# 9주차 — Control I: 샘플링과 Rate 제어

```
[Sensors] → [Estimation] → [Navigation] → [Control] → [Allocation] → [Plant]
                                            ▲ 이번 주
```

목표: 실제 컨트롤러는 **주기 프로그램**(이산 시간)이다. 각속도(rate) PID.
PX4 의 가장 안쪽 루프 — 이번 주 블록은 PX4 v1.16 rate_control.cpp 이식본이다.

## 이론 요약
- sample_time / ZOH / z⁻¹: 6주차에 만든 이산 블록이 실전에 등장
- 캐스케이드의 안쪽부터: rate 루프가 가장 빠르다 (PX4 250Hz~1kHz)
- RateController 내부: P + I(anti-windup) + D(필터링 미분) — ex9 화이트박스로
  더블클릭해 들어가 볼 수 있다
- Ts 가 커지면? 위상 지연 → 성능 저하 → 불안정 (과제)

In [ ]:
# 뷰어 열기 — 뼈대 모델을 브라우저에서 편집한다.
# 완성하면 뷰어의 '저장' 으로 week09_model.json 을 **덮어쓰기** (제출물!)
import sys, webbrowser; sys.path.insert(0, '../..')
from loopsym import load_json, render_html
p = render_html(load_json('week09_model.json'), 'week09_viewer.html',
                title='9주차', t_end=8.0, dt=0.004)
webbrowser.open('file://' + p)

## 실습 — rate 피드백 배선
뼈대는 rate_ctrl 의 각속도 피드백 3줄이 비어 있다 (실행하면 '미연결' 에러가
정확히 어디인지 알려준다). 플랜트의 p/q/r 출력 → rate_ctrl 의 p/q/r 입력.
완성하면: 1초에 요 회전 0.5 rad/s 명령 → 기체가 제자리에서 돌기 시작한다.

In [ ]:
# 노트북에서 바로 실행해 보기 (뷰어 없이) — 저장한 모델을 로드해 시뮬
import sys; sys.path.insert(0, '../..')
from loopsym import load_json, Simulator
bd = load_json('week09_model.json')
sim = Simulator(bd); sim.run(4.0, 0.004)
import matplotlib.pyplot as plt
for b in sim.bd.blocks:
    if type(b).__name__ == 'Scope':
        plt.figure(figsize=(8,2.6)); plt.title(b.name); plt.grid(True)
        for i, lab in enumerate(b.labels): plt.plot(b.t, b.data[i], label=lab)
        plt.legend()

## 과제
1. week09_model.json 완성 (채점기: r 이 0.5 rad/s 추종)
2. rate_ctrl 의 st 를 0.004→0.02→0.05 로 키우며 추종 파형 비교 리포트
   (dt 는 0.004 유지 — st 는 dt 의 정수배).
3. (심화) ex9 화이트박스에서 Rate_Yaw 안에 들어가 LPF(2Hz)의 역할을 설명하라.

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week09.py -q